# 2026-06-16 Day 4: Training Loop And Tiny Overfit

Goal: understand batching, shifted targets, AdamW, evaluation, checkpointing, and tiny-overfit verification.


## How To Use This Reading Notebook

Use this before touching implementation for the day. The goal is not to memorize the paper. The goal is to extract the model of the system you need for today's code and notes.

Workflow:

1. Read only the assigned sections.
2. Fill the mini-lecture answer in your own words.
3. Open the repo files listed in the roadmap.
4. Run the listed checks or record why they skip.
5. Write a short exit ticket before moving on.


## Assigned Reading

| Source | Exact Target | Why It Matters Today |
| --- | --- | --- |
| [nanoGPT](https://github.com/karpathy/nanoGPT) | `train.py` batching, eval, checkpointing | Shows a compact training loop structure for small GPT-style models. |
| [PyTorch AdamW docs](https://docs.pytorch.org/docs/2.12/generated/torch.optim.AdamW.html) | Optimizer behavior and arguments | Explains decoupled weight decay and optimizer parameters. |


## Reading Summary

### nanoGPT `train.py`

The useful pattern is not the exact code; it is the loop structure. Sample batches, run forward/loss, backpropagate, optionally clip gradients, step the optimizer, periodically evaluate, and save checkpoints. Evaluation should run without parameter updates and should use the same loss path as training.

Key takeaway for implementation: a tiny overfit run is a correctness test. If a very small model cannot drive loss down on a tiny repeated batch or shard, the model, loss, optimizer, or data shift is probably wrong.

### PyTorch AdamW

AdamW is Adam with decoupled weight decay. Weight decay is not the same thing as adding L2 directly into the gradient update in classic Adam. In practice, many training scripts put matrices in a decay group and keep bias/norm parameters in a no-decay group.

Key takeaway for implementation: optimizer grouping is part of the training contract, not just an incidental detail.


## Diagram Anchor

![Training step flow](../../assets/figures/training_step_flow.svg)

What to notice: model weights update only after the loss has produced gradients. Checkpointing records enough state to resume or generate later.


## Repo Connection

Open these files after reading:

| File | What To Find |
| --- | --- |
| `scripts/train_dense.py` | Dense wrapper over the shared train script. |
| `scripts/train_moe.py` | CLI args, config construction, eval loop, checkpoint save. |
| `src/moe_transformer_lab/trainable/data.py` | `get_batch` and token window sampling. |
| `src/moe_transformer_lab/trainable/train_utils.py` | optimizer groups, parameter count, checkpoint helpers. |
| `tests/test_tiny_overfit.py` | Minimum expected training behavior. |
| `tests/test_checkpoint.py` | Checkpoint save/load expectations. |


## Shape And Loss Summary

| Item | Shape | Role |
| --- | --- | --- |
| `x` | `(B, T)` | input tokens |
| `y` | `(B, T)` | next-token targets |
| `logits` | `(B, T, V)` | vocabulary scores |
| flattened logits | `(B*T, V)` | cross entropy input |
| flattened targets | `(B*T)` | cross entropy labels |
| `aux_loss` | scalar | MoE router balancing term |


## Mini-Lecture Answer Seed

The model trains by next-token prediction. For each window of tokens, inputs and labels are shifted by one position. Cross entropy compares each `(batch, position)` logits row against the correct next-token ID and averages over positions. The MoE path adds an auxiliary load-balancing term, but it does not change the language-model objective.


## Active Recall

1. Why is a tiny overfit run a useful correctness test?
2. Why should eval use `no_grad`?
3. What does checkpoint resume need besides model weights?
4. Why separate cross entropy and MoE auxiliary loss in logs?

## Exit Ticket

Write one training-loop pseudocode block and list the checkpoint payload fields.
